In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/jvkrishwanth/ica-cleaned/P_27_MW-ica.fif
/kaggle/input/datasets/jvkrishwanth/ica-cleaned/P_14_MW-ica-metadata.mat
/kaggle/input/datasets/jvkrishwanth/ica-cleaned/P_3_MW-clean-epo.fif
/kaggle/input/datasets/jvkrishwanth/ica-cleaned/P_22_MW-clean-epo.fif
/kaggle/input/datasets/jvkrishwanth/ica-cleaned/P_15_MW-clean-epo.fif
/kaggle/input/datasets/jvkrishwanth/ica-cleaned/P_16_MW-ica-metadata.mat
/kaggle/input/datasets/jvkrishwanth/ica-cleaned/P_23_MW-ica.fif
/kaggle/input/datasets/jvkrishwanth/ica-cleaned/P_19_MW-clean-epo.fif
/kaggle/input/datasets/jvkrishwanth/ica-cleaned/P_7_MW-clean-epo.fif
/kaggle/input/datasets/jvkrishwanth/ica-cleaned/P_12_MW-ica-metadata.mat
/kaggle/input/datasets/jvkrishwanth/ica-cleaned/P_25_MW-ica.fif
/kaggle/input/datasets/jvkrishwanth/ica-cleaned/P_14_MW-clean-epo.fif
/kaggle/input/datasets/jvkrishwanth/ica-cleaned/P_11_MW-ica-metadata.mat
/kaggle/input/datasets/jvkrishwanth/ica-cleaned/P_18_MW-ica.fif
/kaggle/input/datasets/jvkrishwant

In [2]:
"""Kaggle-ready EEG feature-family analysis: mind wandering (1) vs focus (0).

Outputs are written to /kaggle/working/feature_family_results by default.
"""

import re
from pathlib import Path

import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import butter, hilbert, sosfiltfilt, welch
from scipy.stats import kurtosis, skew
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedGroupKFold

mne.set_log_level("ERROR")

CHANNELS = [
    "Fp1", "Fp2", "F7", "F3", "Fz", "F4", "F8", "T3", "C3", "Cz", "C4",
    "T4", "T5", "P3", "Pz", "P4", "T6", "O1", "O2",
]
BANDS = {
    "Delta": (1, 4), "Theta": (4, 8), "Alpha": (8, 12),
    "Beta": (13, 30), "Gamma": (30, 45),
}

# Kaggle paths supplied for this analysis. Change only these values if needed.
EPOCHS_DIR = Path(
    "/kaggle/input/datasets/jvkrishwanth/ica-cleaned"
)
LABELS_CSV = Path(
    "/kaggle/input/datasets/jvkrishwanth/df-main-phase1/df_main_phase1.csv"
)
OUTPUT_DIR = Path("/kaggle/working/feature_family_results")


def bandpass(data, sfreq, low, high):
    sos = butter(4, [low / (sfreq / 2), high / (sfreq / 2)], btype="band", output="sos")
    return sosfiltfilt(sos, data, axis=-1)


def shannon_entropy(x, bins=64):
    counts, _ = np.histogram(x, bins=bins)
    p = counts[counts > 0].astype(float)
    if len(p) == 0:
        return 0.0
    p /= p.sum()
    return float(-np.sum(p * np.log2(p)))


def spectral_entropy(psd):
    p = np.asarray(psd, dtype=float)
    p = p[p > 0]
    if len(p) < 2:
        return 0.0
    p /= p.sum()
    return float(-np.sum(p * np.log2(p)) / np.log2(len(p)))


def differential_entropy_std(x, n_segments=6):
    values = []
    for segment in np.array_split(x, n_segments):
        variance = max(np.var(segment), 1e-20)
        values.append(0.5 * np.log(2 * np.pi * np.e * variance))
    return float(np.std(values))


def plv_matrix(data, sfreq, low, high):
    phase = np.angle(hilbert(bandpass(data, sfreq, low, high), axis=-1))
    phase_difference = phase[:, None, :] - phase[None, :, :]
    return np.abs(np.mean(np.exp(1j * phase_difference), axis=-1))


def extended_channel_features(signal, psd, freqs, channel):
    """Requested time-domain, Hjorth, entropy, and spectral-summary features."""
    signal = np.nan_to_num(signal, nan=0.0, posinf=0.0, neginf=0.0)
    variance = np.var(signal)
    d1, d2 = np.diff(signal), np.diff(signal, n=2)
    mobility = np.sqrt(np.var(d1) / max(variance, 1e-20))
    mobility_d1 = np.sqrt(np.var(d2) / max(np.var(d1), 1e-20))
    complexity = mobility_d1 / max(mobility, 1e-20)

    whole = (freqs >= 1) & (freqs <= 45)
    theta = (freqs >= 4) & (freqs < 8)
    alpha = (freqs >= 8) & (freqs < 12)
    band_psd, band_freqs = psd[whole], freqs[whole]
    dominant = float(band_freqs[np.argmax(band_psd)]) if len(band_psd) else 0.0

    return {
        f"{channel}_Mean": float(np.mean(signal)),
        f"{channel}_RMS": float(np.sqrt(np.mean(signal ** 2))),
        f"{channel}_Variance": float(variance),
        f"{channel}_Energy": float(np.sum(signal ** 2)),
        f"{channel}_Peak2Peak": float(np.ptp(signal)),
        f"{channel}_Kurtosis": float(kurtosis(signal, fisher=True, bias=False)),
        f"{channel}_Skewness": float(skew(signal, bias=False)),
        # Activity equals variance; duplicate removal before ML handles this fairly.
        f"{channel}_HjorthActivity": float(variance),
        f"{channel}_HjorthMobility": float(mobility),
        f"{channel}_HjorthComplexity": float(complexity),
        f"{channel}_ShannonEntropy": shannon_entropy(signal),
        f"{channel}_SpectralEntropy": spectral_entropy(band_psd),
        f"{channel}_DifferentialEntropyStd": differential_entropy_std(signal),
        f"{channel}_TotalPower": float(np.trapezoid(band_psd, band_freqs)) if len(band_psd) else 0.0,
        f"{channel}_DominantFreq": dominant,
        f"{channel}_ThetaPower": float(np.mean(psd[theta])) if np.any(theta) else 0.0,
        f"{channel}_AlphaPower": float(np.mean(psd[alpha])) if np.any(alpha) else 0.0,
    }


def window_features(raw_window, sfreq, subject, epoch_index, window_index, label):
    """All old and newly requested features for one 1.5-second EEG window."""
    freqs, psd = welch(raw_window, fs=sfreq, nperseg=min(256, raw_window.shape[-1]), axis=-1)
    envelopes = {
        name: np.abs(hilbert(bandpass(raw_window, sfreq, *limits), axis=-1))
        for name, limits in {"Alpha": BANDS["Alpha"], "Theta": BANDS["Theta"]}.items()
    }
    alpha_corr = np.nan_to_num(np.corrcoef(envelopes["Alpha"]), nan=0.0)
    row = {"Subject": subject, "Epoch_Index": epoch_index, "Window_Index": window_index, "Label": label}

    for i, channel in enumerate(CHANNELS):
        channel_psd = psd[i]
        total_mask = (freqs >= 1) & (freqs <= 45)
        total_power = np.mean(channel_psd[total_mask]) if np.any(total_mask) else 1e-20
        for band, (low, high) in BANDS.items():
            mask = (freqs >= low) & (freqs < high if band != "Gamma" else freqs <= high)
            absolute = np.mean(channel_psd[mask]) if np.any(mask) else 0.0
            row[f"{channel}_{band}_Abs"] = absolute
            row[f"{channel}_{band}_Rel"] = absolute / max(total_power, 1e-20)

        for band in ("Alpha", "Theta"):
            env = envelopes[band][i]
            mean = np.mean(env)
            row[f"{channel}_{band}_Mean"] = float(mean)
            row[f"{channel}_{band}_Var"] = float(np.var(env))
            row[f"{channel}_{band}_CV"] = float(np.std(env) / max(mean, 1e-20))
            threshold = mean + 2 * np.std(env)
            row[f"{channel}_{band}_Bursts"] = float(np.sum(np.diff((env > threshold).astype(int)) == 1))

        others = np.arange(len(CHANNELS)) != i
        row[f"{channel}_Envelope_Sync"] = float(np.mean(alpha_corr[i, others]))
        row.update(extended_channel_features(raw_window[i], channel_psd, freqs, channel))

    # Pairwise functional connectivity (Alpha and Theta PLV).
    for band in ("Alpha", "Theta"):
        matrix = plv_matrix(raw_window, sfreq, *BANDS[band])
        for i in range(len(CHANNELS)):
            for j in range(i + 1, len(CHANNELS)):
                row[f"PLV_{band}_{CHANNELS[i]}_{CHANNELS[j]}"] = matrix[i, j]
    return row


def subject_labels(labels, subject):
    subject_column = labels["Subject"].astype(str).str.replace("P_?", "", regex=True)
    current = labels.loc[subject_column == str(subject)].copy()
    if current.empty:
        return np.array([], dtype=int)
    ordered = current.groupby(["Session", "Epoch_Index"], sort=True)["State"].first()
    return np.where(ordered.astype(str).str.upper().eq("MW"), 1, 0)


def extract_dataset(epochs_dir, labels_csv):
    labels = pd.read_csv(labels_csv)
    required = {"Subject", "Session", "Epoch_Index", "State"}
    missing = required - set(labels.columns)
    if missing:
        raise ValueError(f"The labels CSV is missing columns: {sorted(missing)}")

    rows = []
    files = sorted(Path(epochs_dir).glob("P_*_MW*-epo.fif"))
    if not files:
        raise FileNotFoundError(f"No epoch FIF files found in {epochs_dir}")

    for fif in files:
        match = re.search(r"P_(\d+)_MW", fif.name, flags=re.I)
        if not match:
            continue
        subject = int(match.group(1))
        epochs = mne.read_epochs(fif, verbose=False)
        name_lookup = {name.lower(): name for name in epochs.ch_names}
        usable = [name_lookup[ch.lower()] for ch in CHANNELS if ch.lower() in name_lookup]
        if len(usable) != len(CHANNELS):
            print(f"Skipping P_{subject}: expected all 19 EEG channels.")
            continue
        epochs = epochs.pick(usable)
        epochs.rename_channels({old: new for old, new in zip(epochs.ch_names, CHANNELS) if old != new})
        data, sfreq = epochs.get_data(copy=True), epochs.info["sfreq"]
        y = subject_labels(labels, subject)

        n_epochs = min(len(data), len(y))
        window_length, step = int(1.5 * sfreq), int(0.5 * sfreq)
        for epoch_index in range(n_epochs):
            for window_index, start in enumerate(range(0, data.shape[-1] - window_length + 1, step)):
                window = data[epoch_index, :, start:start + window_length]
                rows.append(window_features(window, sfreq, subject, epoch_index, window_index, y[epoch_index]))
        print(f"P_{subject}: extracted {len(rows)} cumulative windows")
    return pd.DataFrame(rows)


def feature_family(name):
    f = name.lower()
    if f.startswith("plv_") or "envelope_sync" in f:
        return "Functional connectivity / synchrony"
    if "hjorth" in f:
        return "Hjorth features"
    if any(x in f for x in ("spectralentropy", "shannonentropy", "differentialentropy", "sampleentropy", "fractal", "hurst")):
        return "Entropy and complexity"
    if any(x in f for x in ("_alpha_mean", "_alpha_var", "_alpha_cv", "_alpha_bursts", "_theta_mean", "_theta_var", "_theta_cv", "_theta_bursts")):
        return "Oscillatory-envelope dynamics"
    if any(x in f for x in ("_rms", "_mean", "_variance", "_energy", "_peak2peak", "_kurtosis", "_skewness")):
        return "Time-domain statistics"
    if "totalpower" in f or "dominantfreq" in f:
        return "Spectral summary features"
    return "Band-power features"


def analyse(df, output_dir):
    meta = ["Subject", "Epoch_Index", "Window_Index", "Label"]
    X = df.drop(columns=meta).apply(pd.to_numeric, errors="coerce")
    y, groups = df["Label"].to_numpy(), df["Subject"].to_numpy()

    # Removes exact aliases, notably HjorthActivity versus Variance.
    duplicates = X.T.duplicated()
    print(f"Removing {duplicates.sum()} exact duplicate feature columns.")
    X = X.loc[:, ~duplicates]
    feature_names = X.columns.to_numpy()

    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    importances = []
    for fold, (train, test) in enumerate(cv.split(X, y, groups), 1):
        imputer = SimpleImputer(strategy="median")
        x_train = imputer.fit_transform(X.iloc[train])
        x_test = imputer.transform(X.iloc[test])
        selector = SelectKBest(f_classif, k=min(100, x_train.shape[1]))
        x_train = selector.fit_transform(x_train, y[train])
        x_test = selector.transform(x_test)
        selected = feature_names[selector.get_support()]
        model = ExtraTreesClassifier(n_estimators=500, class_weight="balanced", max_features="sqrt", random_state=fold, n_jobs=-1)
        model.fit(x_train, y[train])
        fold_importance = pd.Series(0.0, index=feature_names)
        fold_importance.loc[selected] = model.feature_importances_
        importances.append(fold_importance)

    # Compatible with older Kaggle pandas versions (which lack reset_index(names=...)).
    summary = pd.concat(importances, axis=1).mean(axis=1).rename("Importance").reset_index()
    summary = summary.rename(columns={"index": "Feature"})
    summary["Feature family"] = summary["Feature"].map(feature_family)
    summary = summary.sort_values("Importance", ascending=False)
    family = summary.groupby("Feature family", as_index=False).agg(
        Feature_count=("Feature", "count"), Total_importance=("Importance", "sum"),
        Mean_importance_per_feature=("Importance", "mean"),
    ).sort_values("Mean_importance_per_feature", ascending=False)
    summary.to_csv(output_dir / "feature_importance_by_feature.csv", index=False)
    family.to_csv(output_dir / "feature_importance_by_umbrella.csv", index=False)

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    sns.barplot(data=family, x="Mean_importance_per_feature", y="Feature family", hue="Feature family", legend=False, palette="viridis", ax=axes[0])
    axes[0].set_title("Usefulness per Feature (fair comparison)")
    axes[0].set_xlabel("Mean cross-validated importance")
    sns.barplot(data=family.sort_values("Total_importance", ascending=False), x="Total_importance", y="Feature family", hue="Feature family", legend=False, palette="magma", ax=axes[1])
    axes[1].set_title("Total Contribution by Feature Family")
    axes[1].set_xlabel("Summed cross-validated importance")
    plt.tight_layout()
    plt.savefig(output_dir / "feature_family_comparison.png", dpi=250)
    plt.close()

    top = summary.head(25)
    plt.figure(figsize=(12, 9))
    sns.barplot(data=top, x="Importance", y="Feature", hue="Feature family", dodge=False, palette="tab10")
    plt.title("Top Features: Mind Wandering vs Focus")
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig(output_dir / "top_individual_features.png", dpi=250)
    plt.close()
    print(f"Saved results: {output_dir}")


def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    dataframe = extract_dataset(EPOCHS_DIR, LABELS_CSV)
    dataframe.to_csv(OUTPUT_DIR / "windowed_features_all_families.csv", index=False)
    analyse(dataframe, OUTPUT_DIR)


if __name__ == "__main__":
    main()


P_10: extracted 280 cumulative windows
P_11: extracted 560 cumulative windows
P_12: extracted 840 cumulative windows
P_13: extracted 1120 cumulative windows
P_14: extracted 1400 cumulative windows
P_15: extracted 1680 cumulative windows
P_16: extracted 1960 cumulative windows
P_17: extracted 2240 cumulative windows
P_18: extracted 2520 cumulative windows
P_19: extracted 2800 cumulative windows
P_1: extracted 3080 cumulative windows
P_20: extracted 3360 cumulative windows
P_22: extracted 3640 cumulative windows
P_23: extracted 3920 cumulative windows
P_24: extracted 4200 cumulative windows
P_25: extracted 4480 cumulative windows
P_26: extracted 4760 cumulative windows
P_27: extracted 5040 cumulative windows
P_28: extracted 5320 cumulative windows
P_2: extracted 5600 cumulative windows
P_3: extracted 5880 cumulative windows
P_5: extracted 6160 cumulative windows
P_7: extracted 6440 cumulative windows
P_8: extracted 6720 cumulative windows
P_9: extracted 7000 cumulative windows
Removing 5